# Parda — text model v2, part A (CPU): phone-photo training pages + Tesseract

Makes 6,000 phone-photo versions of Phase 1 **training** pages (2,000 per language; never the 300 benchmark
pages, never the 4 real-world-test layouts) and reads them with Tesseract. Part B (GPU) trains on them.

**Settings:** Accelerator **None (CPU)** — free, no GPU hours · Internet **On**
**Inputs:** your **Parda** (Phase 1) notebook · **Secrets:** `GITHUB_TOKEN`
Run cells 1–4 by hand (~5 min), then **Save Version → Save & Run All** (~1–1.5 h). You can close the browser.


In [ ]:
# 1. Code + secrets
import os, subprocess
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
GH = secrets.get_secret("GITHUB_TOKEN")
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
REPO, CODE = "chetanjagan/Parda", "/tmp/Parda"
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "-q", f"https://{GH}@github.com/{REPO}.git", CODE], check=True)
    subprocess.run(["git", "-C", CODE, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull", "-q", f"https://{GH}@github.com/{REPO}.git", "main"], check=True)
os.chdir(CODE)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)


In [ ]:
# 2. Fonts (for the tests) + Tesseract with Hindi and Kannada
!bash scripts/setup_kaggle.sh
!bash scripts/setup_ocr.sh


In [ ]:
# 3. Phase 1 data
from parda.ocr.run_ocr import find_data, _n_images
DATA = find_data()
print("DATA =", DATA, "| images:", _n_images(DATA))
W = "/kaggle/working"


In [ ]:
# 4. Tests (~1 min): every line ok
!python tests/test_v2.py


In [ ]:
# 5. 6,000 phone-photo training pages (~10 min)
!python -m parda.synth.photo_pages --data {DATA} --out {W}/photo --per_lang 2000 --workers 4
import json, glob
from IPython.display import Image as IPImage, display
for f in sorted(glob.glob(f"{W}/photo/images/*.jpg"))[:2]:
    display(IPImage(f, width=380))


In [ ]:
# 6. Tesseract on all 6,000 photo pages (~1 h on 4 CPUs; resumable)
!python -m parda.pii.ocr_pages --data {W}/photo --engine tesseract --per_lang 0 --bench_of {DATA} --out_dir {W}/photo --workers 4
n = sum(1 for _ in open(f"{W}/photo/ocr_tesseract.jsonl"))
print("OCR pages:", n, "| stats:", json.load(open(f"{W}/photo/stats.json")))
assert n >= 5900, "OCR incomplete - run this cell again (it continues where it stopped)"
